# Chapter 15: Evaluating Agents Properly
## Topic 3: Did It Call the Right Tool (Not Just Reach the Right Label)

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topic 2 established the task-level/step-level distinction generally. This topic takes the single most consequential step-level check for this project's specific agent and builds it out completely: did the agent actually call the correct tool (or correct combination of tools, or correctly call no tool at all) for a given input — evaluated entirely independent of whether the final classification label or answer happened to come out correct.
- Why this specific step-level check deserves its own dedicated topic, beyond Topic 2's general framework: tool-selection correctness is uniquely capable of producing exactly the "right label, wrong reason" pattern this notebook series has repeatedly warned about, in the most direct and immediately actionable form. Chapter 1's classification labels (FD, Non-FD, Multiple Category) could, in principle, be reached correctly by an agent that called the wrong tool, called no tool when one was needed, or called an unnecessary tool — none of which the final label alone would ever reveal.
- The connection to this project's actual, already-built infrastructure is direct and complete: Chapter 9 Topic 1's retrieval-triggering logic, Chapter 10 Topic 6's multi-tool selection, and Chapter 13's confidence-based retrieval decisions are all, precisely, instances of "did it call the right tool" decisions — this topic is where evaluating whether those specific decisions were made correctly, independent of final label correctness, gets its own dedicated, systematic treatment.


### 2. Internal Working — Step by Step

**Evaluating tool-call correctness precisely, independent of final label correctness:**

1. **Define, for each test case, the expected tool-call behavior** — which tool (if any) should have been called, mirroring exactly the kind of labeled test data Chapter 10 Topic 7's triggering-accuracy tests and Chapter 13 Topic 3's catalog-check-decision tests already required constructing.
2. **Run the actual agent (or the relevant decision-making component in isolation) against each test case**, recording which tool it actually chose to call (if any), completely separate from recording what final label or answer it ultimately produced.
3. **Compare the actual tool-call decision against the expected one directly** — this comparison is entirely independent of the final label comparison; a test case can have a correct final label with an incorrect tool-call decision, or an incorrect final label with a perfectly correct tool-call decision, and computing both comparisons separately is precisely what reveals which of these situations, if either, is actually occurring.
4. **Aggregate tool-call correctness into its own accuracy figure, reported alongside (never blended with) final-label accuracy** — exactly Topic 2's task-level/step-level reporting discipline, now specifically instantiated for this one, particularly consequential step-level check.
5. **When tool-call correctness and final-label correctness diverge for a specific test case, investigate that specific case directly** — a correct label reached via an incorrect tool call is a latent risk (Topic 1's core demonstration), while an incorrect label reached via a correct tool call points toward a synthesis/generation-layer problem (Topic 2's second divergence pattern), and distinguishing these requires exactly this kind of separated, case-by-case comparison.


### 3. How This Is Implemented in This Project

- This project's rule-based classifier (`classify_by_keyword_rules()`, Chapter 1) and its GenAI-augmented successor both ultimately produce a final label — but only the agent-based version (from Chapter 3 onward) makes tool-call decisions along the way, meaning this specific evaluation dimension is uniquely relevant to evaluating the agentic, tool-calling layers of this project, not the original, simpler rule-based classifier.
- Concretely, for this project's actual tools — `validate_fd_reference` (Chapter 3, Chapter 10 Topic 3), `search_knowledge_base` (Chapter 9 Topic 3), `lookup_product_catalog` and `check_sender_history` (Chapter 10 Topic 6) — a complete tool-call-correctness evaluation would check, for each test email, whether the agent correctly decided to call each of these tools (or correctly decided not to), independent of whatever final classification label or generated answer resulted.
- This directly reuses labeled test data already built across this notebook series for individual tools (Chapter 9 Topic 1's triggering test set, Chapter 10 Topic 6's multi-tool selection test set, Chapter 13 Topic 3's catalog-check test set) — this topic's contribution is treating tool-call correctness as its own first-class evaluation metric to be tracked continuously, not just as isolated, one-off illustrative demonstrations built separately in each of those earlier topics.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **A correct final label reached via an incorrect tool-call decision is precisely the highest-priority risk this specific metric exists to surface** — Topic 1's own demonstration showed this concretely with a tool-implementation flaw; the equivalent risk at the tool-selection level (the agent calling the wrong tool, or skipping a tool it should have called, yet still landing on a correct-looking final label by coincidence) is exactly as real and exactly as hidden by final-label-only evaluation.
- **Constructing expected-tool-call labels for a real evaluation set requires the same domain expertise and verification discipline Chapter 14 Topic 2 already established for reference answers** — determining what tool (or tools) an agent *should* call for a given real, messy customer email is a genuine judgment call requiring careful reading of the actual content, not a mechanical labeling exercise.
- **Multiple acceptable tool-call sequences (Topic 1's "trajectory isn't always unique" point) apply directly here** — for some test cases, calling `check_sender_history` before or after `validate_fd_reference` might both be entirely acceptable, and the evaluation design needs to accommodate this rather than requiring one single, rigid expected order.
- **Debugging a tool-call-correctness failure requires distinguishing a triggering-decision problem from a tool-selection problem among multiple available tools** — connecting directly to Chapter 9 Topic 1's single-tool triggering discussion versus Chapter 10 Topic 6's genuinely harder multi-tool discrimination problem; these are different failure categories with different fixes (a triggering-logic or schema-clarity fix versus a tool-description-overlap fix), and conflating them wastes debugging effort exactly as this notebook series has repeatedly warned against.
- **Monitoring:** tracking tool-call-correctness accuracy continuously in production (not just at initial evaluation time), broken down per tool, reveals whether a specific tool's triggering reliability is degrading over time — directly connecting to Topic 5's regression-tracking discipline and Chapter 10 Topic 6's own monitoring recommendation for tracking which tool combinations get called together.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Evaluating tool-call correctness for every available tool vs focusing on the highest-stakes tools first:** given the real construction cost of expected-tool-call labels (mirroring Chapter 14 Topic 2's reference-answer construction cost), prioritizing evaluation coverage for the tools whose incorrect triggering carries the highest risk (like `validate_fd_reference`'s cross-customer-data-leak potential, per Topic 1's demonstration) is a more pragmatic starting point than attempting comprehensive coverage across every tool simultaneously.
- **Requiring an exact match on tool-call arguments vs only checking that the correct tool was selected:** checking only tool selection (not argument correctness) is cheaper and catches the coarser, often more consequential failure mode; also checking argument correctness (Chapter 10 Topic 4's schema-quality concern) provides finer diagnostic detail at additional evaluation-construction cost — both are legitimate, complementary checks, and Topic 2's decomposed step-level framework naturally accommodates tracking them as separate sub-metrics rather than forcing a single combined check.
- **How to handle test cases with multiple acceptable tool-call sequences:** requiring the evaluation set to explicitly enumerate all acceptable sequences per test case (rather than just one rigid expected sequence) is more accurate but considerably more expensive to construct — a reasonable middle ground is explicitly marking only the test cases where multiple acceptable sequences genuinely exist and matter, rather than attempting this for every single test case by default.


### 6. Alternatives and When to Use Each

- **Tool-call-correctness evaluation for the highest-risk tools first (this topic's pragmatic recommendation):** the right starting point given real construction cost constraints, prioritizing coverage where an incorrect tool-call decision carries the most serious consequence.
- **Comprehensive tool-call-correctness evaluation across every available tool:** worth building out as the project's evaluation infrastructure matures and evaluation-construction cost is no longer the binding constraint — the eventual target, not necessarily the starting point.
- **Tool-selection-only checks (ignoring argument correctness) vs also checking arguments:** tool-selection-only is a reasonable, cheaper first pass; adding argument-correctness checks (Chapter 10 Topic 4's concern) provides genuinely additional diagnostic value once tool-selection accuracy itself is reasonably well understood and no longer the dominant source of failures.


### 7. Common Mistakes and Production Failures

- Evaluating only final-label accuracy for this project's agentic layers, missing exactly the "correct label via incorrect tool call" risk this topic's dedicated metric exists to surface.
- Requiring a single, rigid expected tool-call sequence per test case when multiple legitimate sequences genuinely exist, incorrectly failing an agent for taking a valid alternative path.
- Not distinguishing a single-tool triggering failure from a multi-tool selection/discrimination failure when debugging a tool-call-correctness problem, applying the wrong category of fix.
- Attempting comprehensive tool-call-correctness evaluation coverage across every tool simultaneously from the start, rather than prioritizing the highest-risk tools first given genuine construction-cost constraints.
- Not monitoring tool-call correctness continuously in production, missing a gradual degradation in a specific tool's triggering reliability until it eventually surfaces as a broader, harder-to-diagnose quality problem.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why is "did it call the right tool" evaluated separately from "did it reach the right final label"?
  A: An agent can reach a correct final label despite calling the wrong tool, calling no tool when one was needed, or calling an unnecessary tool — none of which the final label comparison alone would ever reveal. Evaluating tool-call correctness independently is what surfaces this specific "right label, wrong reason" risk directly.

- Q: What project components does this metric specifically apply to, and why not the original rule-based classifier?
  A: This applies specifically to this project's agentic, tool-calling layers (from Chapter 3 onward) — Chapter 1's original rule-based classifier makes no tool-call decisions at all, so there's no tool-selection behavior to evaluate for that component; this metric is unique to evaluating genuinely agentic behavior.

**Intermediate**

- Q: How does constructing expected-tool-call labels for a real evaluation set differ from constructing expected final-answer labels?
  A: It requires determining not just what the correct final outcome is, but what tool (or tools) an agent *should* call to reach that outcome for a specific, often messy, real customer email — a genuine judgment call requiring careful reading of actual content, mirroring the same domain-expertise-and-verification discipline Chapter 14 Topic 2 established for reference-answer construction, but applied to process rather than outcome.

- Q: Why does the "multiple acceptable tool-call sequences" issue from Topic 1 apply directly to this metric's evaluation design?
  A: For some test cases, several different tool-calling orders or combinations might all be legitimate ways to correctly handle a given email — requiring one single, rigid expected sequence risks incorrectly failing an agent that took a valid alternative approach, so the evaluation design needs to explicitly accommodate cases where multiple acceptable sequences genuinely exist.

**Advanced**

- Q: Design a prioritization strategy for building tool-call-correctness evaluation coverage across this project's four tools, given real construction-cost constraints.
  A: Prioritize based on the consequence of an incorrect tool-call decision for each tool — `validate_fd_reference`'s incorrect triggering carries a demonstrated cross-customer-data-leak risk (Topic 1), making it the highest priority for thorough evaluation coverage. `search_knowledge_base` and `lookup_product_catalog`'s incorrect triggering primarily risks cost and answer quality rather than data exposure, a real but comparatively lower-stakes concern. `check_sender_history`, being proactive and lower-risk if occasionally over- or under-triggered, could reasonably receive the least immediate evaluation investment. This risk-based prioritization makes the most of limited evaluation-construction resources rather than spreading effort evenly regardless of actual stakes.

- Q: A teammate argues that since final-label accuracy is already high, tool-call-correctness evaluation is redundant additional work. How do you respond?
  A: This is precisely the scenario Topic 1 demonstrated concretely can mask a real, serious risk — high final-label accuracy provides no guarantee the underlying tool-call decisions are reliable, and a flaw hidden this way (like an incorrect fuzzy-matching fallback) can persist unnoticed until a specific, previously-untested real-world input exposes it, potentially with serious consequences (a cross-customer data leak, in this project's specific case). Tool-call-correctness evaluation isn't redundant with final-label accuracy — it's the only thing that actually validates whether that final-label accuracy is being achieved through a reliable process.

**Scenario-based**

- Q: Your evaluation shows final-label accuracy at 96% and tool-call correctness at 88%, with the gap concentrated in cases involving `validate_fd_reference`. Walk through your response.
  A: Given `validate_fd_reference`'s demonstrated cross-customer-data-leak risk (Topic 1) when its triggering or matching logic misbehaves, this specific gap deserves immediate, focused investigation — inspect the specific cases where tool-call correctness failed despite a correct final label, checking whether the tool was called with an incorrect reference number, called when it shouldn't have been, or skipped when it should have been called, and specifically checking whether any of these failures could plausibly have produced a cross-customer data exposure in a slightly different, real-world case. This gap is not an acceptable trade-off for high final-label accuracy — it's exactly the latent risk this project's evaluation practice needs to catch and address before it manifests in production.

**Follow-up questions to expect:**

- "How would you validate that your expected-tool-call labels are themselves correct, given the judgment involved in constructing them?"
- "What would you do if tool-call correctness for a specific tool couldn't be improved through prompting alone?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This topic's metric is a specific, high-value instance of Topic 2's general step-level metrics framework** — recognizing "did it call the right tool" as one particular, especially consequential step-level check (rather than a separate, unrelated evaluation technique) connects this topic directly back to the general framework Topic 2 established.
- **The risk this metric specifically targets — a correct label masking an incorrect, potentially unsafe process — is the same underlying concern that motivated Chapter 9 Topic 4's exact-match-only design principle for customer-record lookups in the first place** — this topic's evaluation practice is what would actually catch a violation of that design principle if one were ever introduced, closing the loop between a design principle and the evaluation practice that verifies it's being followed.
- **This topic sets up Topics 4 and 5 directly**: a repeatable evaluation harness (Topic 4) needs tool-call-correctness checks like this one as one of its core, standing components, and regression tracking (Topic 5) needs exactly this kind of metric tracked over time to detect when a pipeline change degrades tool-calling reliability specifically, not just final-label accuracy.

### 10. Quick Revision Summary (for last-minute interview prep)

> Evaluating whether an agent called the right tool, independent of whether it reached the right final label, is the single most consequential step-level check (Topic 2's framework) this project's agentic layers need, because a correct final label can mask an incorrect, potentially unsafe tool-call decision — exactly Topic 1's "right answer, wrong process" risk, now applied specifically to tool selection. This requires labeled test data specifying the expected tool-call behavior (which tool, or correctly no tool) for each test case, evaluated entirely separately from final-label correctness, and accommodating cases where multiple tool-call sequences are genuinely, legitimately acceptable rather than requiring one single rigid expected path. Given the real cost of constructing this kind of labeled data, prioritizing evaluation coverage for the highest-risk tools first — `validate_fd_reference`, given its demonstrated cross-customer-data-leak potential when triggering or matching misbehaves — is more pragmatic than attempting comprehensive coverage across every tool simultaneously. This metric is what actually closes the loop on design principles like Chapter 9 Topic 4's exact-match-only rule: without evaluating tool-call correctness specifically, a violation of that principle could persist unnoticed behind consistently high final-label accuracy, exactly the latent risk this topic's dedicated evaluation practice exists to catch.


### Module 1: A Real, Simulated Agent Making Tool-Call Decisions Across Multiple Tools

Build a simplified multi-tool agent decision function, and a labeled test set specifying the EXPECTED tool-call decision for each case, independent of final label.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Multi-tool agent decisions + expected-tool-call labels
# ------------------------------------------------------------------

import re

def agent_decide_tools(email_text: str) -> list:
    """Simplified agent tool-selection logic across THREE tools --
    mirroring Chapter 10 Topic 6's multi-tool selection pattern."""
    tools_called = []
    if re.search(r'\b[A-Z]{2}\d{4}FD\d{4}\b', email_text):
        tools_called.append("validate_fd_reference")
    if "smart saver" in email_text.lower() or "senior citizen fd" in email_text.lower():
        tools_called.append("lookup_product_catalog")
    if "penalty" in email_text.lower() or "withdraw" in email_text.lower():
        tools_called.append("search_knowledge_base")
    return tools_called


# labeled test set: (email, expected_tools_called) -- INDEPENDENT of
# whatever final label the classifier assigns
LABELED_TOOL_CALL_SET = [
    ("What is the penalty for withdrawing my FD BJ2019FD7717?",
     {"validate_fd_reference", "search_knowledge_base"}),
    ("Tell me about the Smart Saver FD interest rate.",
     {"lookup_product_catalog"}),
    ("My loan EMI bounced, please help.", set()),
    ("Check my FD BJ2022FD6918 status please.", {"validate_fd_reference"}),
    ("What is the penalty for premature withdrawal in general?",
     {"search_knowledge_base"}),
]

print("=" * 70)
print("MODULE 1: MULTI-TOOL AGENT + EXPECTED-TOOL-CALL LABELS")
print("=" * 70)
for email, expected_tools in LABELED_TOOL_CALL_SET:
    actual_tools = set(agent_decide_tools(email))
    print(f"\nEmail: '{email[:55]}...'")
    expected_display = expected_tools if expected_tools else "(none)"
    print(f"  Expected tools: {expected_display}")
    actual_display = actual_tools if actual_tools else "(none)"
    print(f"  Actual tools:   {actual_display}")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: MULTI-TOOL AGENT + EXPECTED-TOOL-CALL LABELS

Email: 'What is the penalty for withdrawing my FD BJ2019FD7717?...'
  Expected tools: {'search_knowledge_base', 'validate_fd_reference'}
  Actual tools:   {'search_knowledge_base', 'validate_fd_reference'}

Email: 'Tell me about the Smart Saver FD interest rate....'
  Expected tools: {'lookup_product_catalog'}
  Actual tools:   {'lookup_product_catalog'}

Email: 'My loan EMI bounced, please help....'
  Expected tools: (none)
  Actual tools:   (none)

Email: 'Check my FD BJ2022FD6918 status please....'
  Expected tools: {'validate_fd_reference'}
  Actual tools:   {'validate_fd_reference'}

Email: 'What is the penalty for premature withdrawal in general...'
  Expected tools: {'search_knowledge_base'}
  Actual tools:   {'search_knowledge_base'}

Module 1 complete. Run Module 2 next.


### Module 2: Tool-Call Correctness, Measured Independent of Final Label

Compute tool-call-correctness accuracy directly, and show a real case where final label is correct but tool-call decision is wrong -- the exact risk this topic targets.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Tool-call correctness, independent of final label
# ------------------------------------------------------------------

def classify_final_label(email_text: str) -> str:
    """A SIMPLE final-label classifier, deliberately using a DIFFERENT,
    cruder signal than the tool-selection logic -- so we can show a
    REAL case where final label is right but tool-call decision differs."""
    text_lower = email_text.lower()
    if "loan" in text_lower or "emi" in text_lower:
        return "Non-FD"
    return "FD"  # crude: anything else defaults to FD


# a DELIBERATE test case: final label will be CORRECT, but the crude
# classify_final_label function reaches it WITHOUT the agent needing
# to have called any tool correctly at all -- demonstrating that label
# correctness and tool-call correctness are genuinely independent
LABELED_TOOL_CALL_SET_WITH_LABELS = [
    ("What is the penalty for withdrawing my FD BJ2019FD7717?",
     {"validate_fd_reference", "search_knowledge_base"}, "FD"),
    ("Tell me about the Smart Saver FD interest rate.",
     {"lookup_product_catalog"}, "FD"),
    ("My loan EMI bounced, please help.", set(), "Non-FD"),
    ("Check my FD BJ2022FD6918 status please.", {"validate_fd_reference"}, "FD"),
    # DELIBERATE case: label will be correct ("FD"), but our simple
    # agent_decide_tools() function will WRONGLY miss the product name
    # due to a typo ("smrt saver" instead of "smart saver")
    ("Tell me about the smrt saver fd rate please.", {"lookup_product_catalog"}, "FD"),
]

print("=" * 70)
print("MODULE 2: TOOL-CALL CORRECTNESS vs FINAL-LABEL CORRECTNESS")
print("=" * 70)

tool_call_correct_count = 0
label_correct_count = 0
divergent_cases = []

for email, expected_tools, expected_label in LABELED_TOOL_CALL_SET_WITH_LABELS:
    actual_tools = set(agent_decide_tools(email))
    actual_label = classify_final_label(email)

    tool_call_correct = actual_tools == expected_tools
    label_correct = actual_label == expected_label

    tool_call_correct_count += tool_call_correct
    label_correct_count += label_correct

    print(f"\nEmail: '{email[:55]}...'")
    expected_disp = expected_tools if expected_tools else "(none)"
    actual_disp = actual_tools if actual_tools else "(none)"
    print(f"  Tool-call correct? {tool_call_correct} (expected={expected_disp}, actual={actual_disp})")
    print(f"  Label correct?     {label_correct} (expected={expected_label}, actual={actual_label})")

    if label_correct and not tool_call_correct:
        divergent_cases.append(email)
        print(f"  *** DIVERGENCE: label is CORRECT despite an INCORRECT tool-call")
        print(f"      decision -- exactly the risk this topic's metric exists to catch. ***")

total = len(LABELED_TOOL_CALL_SET_WITH_LABELS)
tool_call_accuracy = tool_call_correct_count / total
label_accuracy = label_correct_count / total

print(f"\nTool-call correctness accuracy: {tool_call_accuracy:.2f}")
print(f"Final-label accuracy:           {label_accuracy:.2f}")
print(f"\nDivergent cases (correct label, WRONG tool-call decision): {len(divergent_cases)}")
for case in divergent_cases:
    print(f"  - '{case}'")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: TOOL-CALL CORRECTNESS vs FINAL-LABEL CORRECTNESS

Email: 'What is the penalty for withdrawing my FD BJ2019FD7717?...'
  Tool-call correct? True (expected={'search_knowledge_base', 'validate_fd_reference'}, actual={'search_knowledge_base', 'validate_fd_reference'})
  Label correct?     True (expected=FD, actual=FD)

Email: 'Tell me about the Smart Saver FD interest rate....'
  Tool-call correct? True (expected={'lookup_product_catalog'}, actual={'lookup_product_catalog'})
  Label correct?     True (expected=FD, actual=FD)

Email: 'My loan EMI bounced, please help....'
  Tool-call correct? True (expected=(none), actual=(none))
  Label correct?     True (expected=Non-FD, actual=Non-FD)

Email: 'Check my FD BJ2022FD6918 status please....'
  Tool-call correct? True (expected={'validate_fd_reference'}, actual={'validate_fd_reference'})
  Label correct?     True (expected=FD, actual=FD)

Email: 'Tell me about the smrt saver fd rate please....'
  Tool-call correct? False (expected={'

### Module 3: Risk-Based Prioritization of Tool-Call Evaluation Coverage

Compute a real, evidence-based prioritization of which tools deserve the most evaluation investment, based on the actual consequence of an incorrect tool-call decision for each.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Risk-based prioritization for evaluation investment
# ------------------------------------------------------------------

TOOL_RISK_PROFILE = {
    "validate_fd_reference": {
        "consequence_of_error": "cross-customer data leak (Topic 1's demonstrated risk)",
        "risk_score": 9,  # out of 10 -- highest, given the demonstrated real risk
    },
    "lookup_product_catalog": {
        "consequence_of_error": "wrong product figures stated to customer (answer-quality risk)",
        "risk_score": 6,
    },
    "search_knowledge_base": {
        "consequence_of_error": "generic or incomplete policy answer (answer-quality risk)",
        "risk_score": 5,
    },
    "check_sender_history": {
        "consequence_of_error": "missed or unnecessary context (low-stakes, proactive tool)",
        "risk_score": 3,
    },
}

print("=" * 70)
print("MODULE 3: RISK-BASED EVALUATION-COVERAGE PRIORITIZATION")
print("=" * 70)

# real, computed prioritization -- sorted by risk score, highest first
prioritized_tools = sorted(TOOL_RISK_PROFILE.items(), key=lambda x: x[1]["risk_score"], reverse=True)

print(f"\n{'Priority':<10} | {'Tool':<25} | {'Risk Score':>10} | Consequence of Error")
print("-" * 95)
for priority, (tool_name, profile) in enumerate(prioritized_tools, start=1):
    consequence = profile["consequence_of_error"]
    risk = profile["risk_score"]
    print(f"{priority:<10} | {tool_name:<25} | {risk:>10} | {consequence}")

print(f"\nCONCLUSION: given limited evaluation-construction resources,")
print(f"'{prioritized_tools[0][0]}' should receive the FIRST and MOST THOROUGH")
print(f"tool-call-correctness evaluation coverage, given its demonstrated,")
print(f"highest-consequence failure mode (Topic 1's real cross-customer")
print(f"data leak demonstration) -- exactly the evidence-based prioritization")
print(f"this topic's theory recommends, rather than spreading evaluation")
print(f"effort evenly across all tools regardless of actual risk.")

print("\nModule 3 complete. All Chapter 15 Topic 3 modules done.")
print()
print("=" * 70)
print("CHAPTER 15 TOPIC 3 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Tool-call correctness is measured COMPLETELY INDEPENDENTLY of final
  label correctness -- demonstrated concretely: a real test case showed
  a CORRECT final label reached via an INCORRECT tool-call decision
  (a typo'd product name missed by the tool-selection logic).

  This divergence (correct label, wrong tool-call) is EXACTLY the
  highest-priority risk this metric exists to catch -- final-label-only
  evaluation would have completely missed it.

  Risk-based prioritization of evaluation coverage is a REAL, COMPUTED
  decision, not arbitrary -- validate_fd_reference ranked highest given
  its DEMONSTRATED (Topic 1) cross-customer data leak consequence.

  This metric is what actually verifies Chapter 9 Topic 4's exact-match
  design principle is being followed in practice, not just assumed.
""")


MODULE 3: RISK-BASED EVALUATION-COVERAGE PRIORITIZATION

Priority   | Tool                      | Risk Score | Consequence of Error
-----------------------------------------------------------------------------------------------
1          | validate_fd_reference     |          9 | cross-customer data leak (Topic 1's demonstrated risk)
2          | lookup_product_catalog    |          6 | wrong product figures stated to customer (answer-quality risk)
3          | search_knowledge_base     |          5 | generic or incomplete policy answer (answer-quality risk)
4          | check_sender_history      |          3 | missed or unnecessary context (low-stakes, proactive tool)

CONCLUSION: given limited evaluation-construction resources,
'validate_fd_reference' should receive the FIRST and MOST THOROUGH
tool-call-correctness evaluation coverage, given its demonstrated,
highest-consequence failure mode (Topic 1's real cross-customer
data leak demonstration) -- exactly the evidence-based priori